# Multilingual Fake News Detector — Colab training

Fine-tune `bert-base-multilingual-cased` on six languages (English, Urdu,
Spanish, German, Chinese, Korean) using Colab's **free GPU**, then download the
exported TensorFlow SavedModel to drop into `./saved_model/` locally.

**Before running:** Runtime ▸ Change runtime type ▸ Hardware accelerator = **GPU (T4)**.

This notebook runs the same `data_prep.py` and `train.py` that ship in the repo —
the only difference from local training is the GPU.

## 1. Get the project files

Either clone your GitHub fork (set `REPO_URL`) **or** comment the clone out and
use the Files panel to upload `data_prep.py`, `train.py`, and the `data/seed/`
CSVs manually.

In [1]:
REPO_URL = "https://github.com/<your-username>/Fake-News-Detection.git"  # <-- edit me

import os
if REPO_URL and "<your-username>" not in REPO_URL:
    !git clone $REPO_URL project
    %cd project
else:
    print("Set REPO_URL above, or upload data_prep.py, train.py and data/seed/*.csv "
          "via the Files panel, then skip this cell.")

Set REPO_URL above, or upload data_prep.py, train.py and data/seed/*.csv via the Files panel, then skip this cell.


## 2. Install dependencies

Colab already has a recent TensorFlow with GPU support; we only add the
training/data libraries.

In [2]:
# Colab ships a newer pyarrow that breaks datasets 2.14.6 (PyExtensionType removed),
# so pin pyarrow==14.0.2 and the matching huggingface_hub. Restart runtime if prompted.
!pip install -q "transformers==4.35.0" "datasets==2.14.6" "pyarrow==14.0.2" "huggingface_hub==0.17.3" scikit-learn pandas openpyxl sentencepiece

import tensorflow as tf
print("TF", tf.__version__, "| GPUs:", tf.config.list_physical_devices('GPU'))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.1/123.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 493.7/493.7 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.0/38.0 MB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.0/295.0 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 118.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 kB 15.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 1.14.0 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 0.17.3 which is 


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelapp.py", line 712, in start
    self.io_loop.start()
  File "/usr/local/lib/python3.12/dist-package

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelapp.py", line 712, in start
    self.io_loop.start()
  File "/usr/local/lib/python3.12/dist-package

AttributeError: _ARRAY_API not found

TF 2.20.0 | GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 3. Build the corpus

Downloads the English/Urdu/Spanish datasets from the HuggingFace Hub and merges
them with the committed German/Chinese/Korean seed sets. Drop `--subset` for the
full corpus (recommended on GPU).

In [3]:
!python data_prep.py            # add e.g. --subset 2000 for a quick run

python3: can't open file '/content/data_prep.py': [Errno 2] No such file or directory


## 4. Train + export the SavedModel

On a T4 GPU a few epochs over the full corpus take only minutes.

In [4]:
!python train.py --csv data/train.csv --epochs 3 --batch-size 16

python3: can't open file '/content/train.py': [Errno 2] No such file or directory


## 5. (Optional) Evaluate

In [5]:
!python evaluate.py --csv data/test.csv

python3: can't open file '/content/evaluate.py': [Errno 2] No such file or directory


## 6. Download the SavedModel

Zips `saved_model/` and downloads it. Unzip it into the repo root locally so
that `./saved_model/saved_model.pb` exists, then run `app.py` / `api.py`.

In [6]:
import shutil
from google.colab import files
shutil.make_archive("saved_model", "zip", "saved_model")
files.download("saved_model.zip")

FileNotFoundError: [Errno 2] No such file or directory: 'saved_model'